# Pelatihan konteks lanjutan dengan cache prefix
Seluruh data merupakan training/development. Hasil notebook 26 yang belum memenuhi syarat dipertahankan. Cache prefix beku diverifikasi terhadap forward penuh sebelum gradien. Kandidat hanya boleh diekspor bila seluruh syarat yang sama terpenuhi. Tidak ada inferensi penerimaan baru.


In [ ]:
from pathlib import Path
from datetime import datetime,timezone
import os,sys,json,subprocess
ROOT=Path(os.environ['SLR09_PACKAGE_ROOT']).resolve()
WORK=Path(os.environ['SLR09_WORK_DIR']).resolve()
PREVIOUS=Path(os.environ['SLR09_PREVIOUS_TRAINING']).resolve()
CACHE=Path(os.environ['SLR09_PREFIX_CACHE']).resolve()
WARM=Path(os.environ['SLR09_WARM_START']).resolve()
def command(argv):
    proc=subprocess.Popen([sys.executable,*map(str,argv)],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    for line in proc.stdout:print(line,end='',flush=True)
    proc.wait()
    if proc.returncode:raise RuntimeError('Actual command failed: '+str(argv[0]))
if not (CACHE/'manifest.json').exists():
    command([ROOT/'code/quality_fix/cache_frozen_prefix_v083.py','--root',ROOT,'--training',PREVIOUS,'--warm',WARM,'--output',CACHE])
assert json.loads((CACHE/'manifest.json').read_text())['status']=='SUCCEEDED'
OUT=WORK/('cached_context_training_v083_'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ'))
print('ACTUAL_CACHED_TRAINING_OUTPUT',OUT,flush=True)
command([ROOT/'code/quality_fix/train_cached_context_v083.py','--root',ROOT,'--previous',PREVIOUS,
    '--cache',CACHE,'--original-warm',WARM,'--output',OUT])
state=json.loads((OUT/'run_state.json').read_text())
assert state['status']=='SUCCEEDED' and state['all_known_constraints_met']
print(json.dumps(state,indent=2))
